In [1]:
# -*- coding: utf-8 -*-
"""
根据 RF 正式模型结果中的 TreeSHAP 重要性，绘制双层环形图和分类占比柱状图。

输入文件：
E:/excel/FILES/博士申请/第二篇论文/脚本/Python/importance/
RF_formal_model_results.xlsx

输入工作表：SHAP_importance

原始配色：
- 外环：Matplotlib tab10 前三种颜色
- 内环及柱状图：Matplotlib Pastel1 前五种颜色

输出：
1. SUNNY 单独图（PNG、PDF）
2. HEATWAVE 单独图（PNG、PDF）
3. 两情景上下排列综合图（PNG、PDF）
4. 分类汇总 CSV 和 Excel
"""

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

warnings.filterwarnings("ignore")


# =============================================================================
# 1. 路径与参数
# =============================================================================
INPUT_PATH = Path(
    r"E:\excel\FILES\博士申请\第二篇论文\脚本\Python\importance"
    r"\RF_formal_model_results.xlsx"
)
INPUT_SHEET = "SHAP_importance"

OUTPUT_DIR = Path(
    r"E:\excel\FILES\博士申请\第二篇论文\脚本\Python\importance\cycle"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGETS = ["SUNNY", "HEATWAVE"]

TARGET_LABELS = {
    "SUNNY": "Sunny",
    "HEATWAVE": "Heatwave",
}

TARGET_FULL_LABELS = {
    "SUNNY": "Sunny-day condition",
    "HEATWAVE": "Heatwave condition",
}


# =============================================================================
# 2. 变量分类
# =============================================================================
OUTER_GROUPS = {
    "External landscapes": [
        "TCC_b90m", "GCI_b90m", "BCI_b90m", "WCI_b90m",
        "RCI_b90m", "BAH_b90m", "BHSD_b90m",
    ],
    "Internal landscapes": [
        "TCC", "GCI", "BCI", "WCI", "RCI", "BAH", "BHSD",
    ],
    "Control variables": [
        "Shape_Area", "jungong", "DIST",
    ],
}

INNER_GROUPS = {
    "External 2D characteristics": [
        "TCC_b90m", "GCI_b90m", "BCI_b90m", "WCI_b90m", "RCI_b90m",
    ],
    "External 3D characteristics": [
        "BAH_b90m", "BHSD_b90m",
    ],
    "Internal 2D characteristics": [
        "TCC", "GCI", "BCI", "WCI", "RCI",
    ],
    "Internal 3D characteristics": [
        "BAH", "BHSD",
    ],
    "Control variables": [
        "Shape_Area", "jungong", "DIST",
    ],
}

BAR_LABELS = {
    "External 2D characteristics": "External 2D",
    "External 3D characteristics": "External 3D",
    "Internal 2D characteristics": "Internal 2D",
    "Internal 3D characteristics": "Internal 3D",
    "Control variables": "Control variables",
}


# =============================================================================
# 3. 原始配色方案
# =============================================================================
def get_plot_colors() -> tuple[list, list]:
    """返回原始代码中的 tab10 + Pastel1 配色。"""
    outer_cmap = plt.get_cmap("tab10")
    inner_cmap = plt.get_cmap("Pastel1")

    outer_colors = [outer_cmap(i) for i in range(3)]
    inner_colors = [inner_cmap(i) for i in range(5)]

    return outer_colors, inner_colors


# =============================================================================
# 4. 数据读取与检查
# =============================================================================
def read_shap_importance() -> pd.DataFrame:
    """读取 SHAP_importance 工作表并规范字段类型。"""
    if not INPUT_PATH.exists():
        raise FileNotFoundError(f"找不到输入文件：\n{INPUT_PATH}")

    try:
        data = pd.read_excel(
            INPUT_PATH,
            sheet_name=INPUT_SHEET,
            engine="openpyxl",
        )
    except ValueError as error:
        raise ValueError(
            f"Excel 文件中找不到工作表：{INPUT_SHEET}"
        ) from error

    data = data.dropna(how="all").copy()
    data.columns = data.columns.astype(str).str.strip()

    for column in ["LST", "Variable", "Variable_Label"]:
        if column in data.columns:
            data[column] = data[column].astype(str).str.strip()

    if "LST" in data.columns:
        data["LST"] = data["LST"].str.upper()

    for column in ["MeanAbsSHAP", "RelativeImportance_percent"]:
        if column in data.columns:
            data[column] = pd.to_numeric(data[column], errors="coerce")

    data.replace([np.inf, -np.inf], np.nan, inplace=True)
    return data


def validate_group_definitions() -> None:
    """检查分类字典是否存在重复或遗漏。"""
    outer_variables = [v for values in OUTER_GROUPS.values() for v in values]
    inner_variables = [v for values in INNER_GROUPS.values() for v in values]

    outer_duplicates = sorted(
        {v for v in outer_variables if outer_variables.count(v) > 1}
    )
    inner_duplicates = sorted(
        {v for v in inner_variables if inner_variables.count(v) > 1}
    )

    if outer_duplicates:
        raise ValueError(f"外环分类中存在重复变量：{outer_duplicates}")
    if inner_duplicates:
        raise ValueError(f"内环分类中存在重复变量：{inner_duplicates}")
    if set(outer_variables) != set(inner_variables):
        raise ValueError("外环与内环包含的变量集合不一致。")


def validate_input(data: pd.DataFrame) -> None:
    """检查输入字段、情景、变量及数值完整性。"""
    required_columns = {
        "LST",
        "Variable",
        "Variable_Label",
        "MeanAbsSHAP",
        "RelativeImportance_percent",
    }

    missing_columns = sorted(required_columns.difference(data.columns))
    if missing_columns:
        raise KeyError(
            "SHAP_importance 工作表缺少以下字段：\n"
            + "\n".join(missing_columns)
        )

    observed_targets = set(data["LST"].dropna().astype(str).str.upper())
    missing_targets = sorted(set(TARGETS).difference(observed_targets))
    if missing_targets:
        raise KeyError(
            "SHAP_importance 工作表缺少以下情景：\n"
            + "\n".join(missing_targets)
        )

    required_variables = {
        variable
        for variable_list in OUTER_GROUPS.values()
        for variable in variable_list
    }

    for target in TARGETS:
        target_rows = data.loc[data["LST"] == target].copy()

        duplicates = target_rows.loc[
            target_rows["Variable"].duplicated(keep=False), "Variable"
        ].unique()
        if len(duplicates) > 0:
            raise ValueError(
                f"{target} 中以下变量出现重复记录：\n"
                + "\n".join(sorted(map(str, duplicates)))
            )

        current_variables = set(target_rows["Variable"].dropna().astype(str))
        missing_variables = sorted(required_variables.difference(current_variables))
        if missing_variables:
            raise KeyError(
                f"{target} 缺少以下变量：\n"
                + "\n".join(missing_variables)
            )

    invalid_shap = data["MeanAbsSHAP"].isna() | ~np.isfinite(data["MeanAbsSHAP"])
    if invalid_shap.any():
        invalid_rows = data.loc[
            invalid_shap,
            ["LST", "Variable", "MeanAbsSHAP"],
        ]
        raise ValueError(
            "MeanAbsSHAP 中存在无法识别的值：\n"
            + invalid_rows.to_string(index=False)
        )

    if (data["MeanAbsSHAP"] < 0).any():
        raise ValueError("MeanAbsSHAP 中存在负值，请检查输入数据。")


# =============================================================================
# 5. 分类汇总
# =============================================================================
def aggregate_groups(
    data: pd.DataFrame,
    target: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """按 MeanAbsSHAP 求和后，重新计算分类百分比。"""
    target_data = (
        data.loc[data["LST"] == target]
        .copy()
        .set_index("Variable")
    )

    total_shap = float(target_data["MeanAbsSHAP"].sum())
    if not np.isfinite(total_shap) or total_shap <= 0:
        raise ValueError(f"{target} 的 MeanAbsSHAP 总和无效：{total_shap}")

    def make_summary(groups: dict[str, list[str]], level: str) -> pd.DataFrame:
        rows = []
        for group_name, variables in groups.items():
            group_shap = float(target_data.loc[variables, "MeanAbsSHAP"].sum())
            rows.append(
                {
                    "LST": target,
                    "Level": level,
                    "Group": group_name,
                    "MeanAbsSHAP_sum": group_shap,
                    "Percentage": group_shap / total_shap * 100,
                }
            )
        return pd.DataFrame(rows)

    outer_df = make_summary(OUTER_GROUPS, "Outer")
    inner_df = make_summary(INNER_GROUPS, "Inner")

    for level_name, result in [("外环", outer_df), ("内环", inner_df)]:
        percentage_sum = result["Percentage"].sum()
        if not np.isclose(percentage_sum, 100.0, atol=1e-6):
            raise ValueError(
                f"{target} {level_name}分类百分比之和不等于 100%："
                f"{percentage_sum:.10f}"
            )

    return outer_df, inner_df


# =============================================================================
# 6. 绘图
# =============================================================================
def draw_donut_and_bar(
    fig: plt.Figure,
    grid_cell,
    outer_df: pd.DataFrame,
    inner_df: pd.DataFrame,
    target: str,
) -> None:
    """在一个 GridSpec 区域内绘制双层环形图和水平柱状图。"""
    subgrid = grid_cell.subgridspec(
        1,
        2,
        width_ratios=[1.18, 1.00],
        wspace=0.35,
    )

    ax_donut = fig.add_subplot(subgrid[0, 0])
    ax_bar = fig.add_subplot(subgrid[0, 1])

    outer_values = outer_df["Percentage"].to_numpy(dtype=float)
    outer_names = outer_df["Group"].tolist()
    inner_values = inner_df["Percentage"].to_numpy(dtype=float)
    inner_names = inner_df["Group"].tolist()

    # 使用原始配色
    outer_colors, inner_colors = get_plot_colors()

    outer_labels = [
        f"{name}\n{value:.1f}%"
        for name, value in zip(outer_names, outer_values)
    ]

    ax_donut.pie(
        outer_values,
        radius=1.00,
        labels=outer_labels,
        labeldistance=1.13,
        colors=outer_colors,
        startangle=90,
        counterclock=False,
        wedgeprops={
            "width": 0.27,
            "edgecolor": "white",
            "linewidth": 1.5,
        },
        textprops={
            "fontsize": 10.5,
            "fontweight": "bold",
            "ha": "center",
        },
    )

    ax_donut.pie(
        inner_values,
        radius=0.71,
        labels=None,
        colors=inner_colors,
        startangle=90,
        counterclock=False,
        wedgeprops={
            "width": 0.27,
            "edgecolor": "white",
            "linewidth": 1.5,
        },
    )

    ax_donut.text(
        0,
        0.05,
        TARGET_LABELS[target],
        ha="center",
        va="center",
        fontsize=15,
        fontweight="bold",
    )
    ax_donut.text(
        0,
        -0.12,
        "SHAP",
        ha="center",
        va="center",
        fontsize=10,
        fontweight="bold",
        color="#666666",
    )
    ax_donut.set_aspect("equal")

    legend_handles = [
        Patch(
            facecolor=inner_colors[i],
            edgecolor="white",
            label=f"{name}: {inner_values[i]:.1f}%",
        )
        for i, name in enumerate(inner_names)
    ]

    ax_donut.legend(
        handles=legend_handles,
        title="Inner-ring categories",
        loc="upper center",
        bbox_to_anchor=(0.50, -0.08),
        ncol=1,
        frameon=False,
        fontsize=9.2,
        title_fontsize=10.2,
    )

    # 与原代码一致：反转顺序，使第一类显示在最上方
    bar_df = inner_df.copy()
    bar_df["DisplayLabel"] = bar_df["Group"].map(BAR_LABELS)
    bar_df = bar_df.iloc[::-1].reset_index(drop=True)
    bar_colors = inner_colors[::-1]

    bars = ax_bar.barh(
        bar_df["DisplayLabel"],
        bar_df["Percentage"],
        color=bar_colors,
        edgecolor="black",
        linewidth=0.65,
        alpha=0.92,
        height=0.66,
    )

    max_percent = float(bar_df["Percentage"].max())
    x_limit = max(max_percent * 1.30, max_percent + 3.0)
    ax_bar.set_xlim(0, x_limit)

    for bar, value in zip(bars, bar_df["Percentage"]):
        ax_bar.text(
            bar.get_width() + max_percent * 0.025,
            bar.get_y() + bar.get_height() / 2,
            f"{value:.1f}%",
            va="center",
            ha="left",
            fontsize=10.5,
            fontweight="bold",
        )

    ax_bar.set_xlabel(
        "Relative importance (%)",
        fontsize=12,
        fontweight="bold",
        labelpad=8,
    )
    ax_bar.set_ylabel("")
    ax_bar.set_title(
        "Grouped SHAP importance",
        fontsize=13,
        fontweight="bold",
        pad=14,
    )

    ax_bar.tick_params(
        axis="both",
        labelsize=10.5,
        width=1.1,
        length=4.5,
        direction="out",
    )

    for tick_label in ax_bar.get_xticklabels() + ax_bar.get_yticklabels():
        tick_label.set_fontweight("bold")

    ax_bar.spines["top"].set_visible(False)
    ax_bar.spines["right"].set_visible(False)
    ax_bar.spines["left"].set_linewidth(1.15)
    ax_bar.spines["bottom"].set_linewidth(1.15)
    ax_bar.grid(False)


def save_single_figure(
    outer_df: pd.DataFrame,
    inner_df: pd.DataFrame,
    target: str,
) -> None:
    """保存一个天气情景的单独图。"""
    fig = plt.figure(figsize=(13.2, 7.0), dpi=180)
    grid = fig.add_gridspec(1, 1)

    draw_donut_and_bar(
        fig=fig,
        grid_cell=grid[0, 0],
        outer_df=outer_df,
        inner_df=inner_df,
        target=target,
    )

    fig.suptitle(
        f"{TARGET_FULL_LABELS[target]}: grouped SHAP importance",
        fontsize=16,
        fontweight="bold",
        y=0.985,
    )

    fig.subplots_adjust(
        left=0.05,
        right=0.97,
        top=0.89,
        bottom=0.22,
    )

    filename_stem = f"{target}_SHAP_grouped_double_donut_bar"
    fig.savefig(
        OUTPUT_DIR / f"{filename_stem}.png",
        dpi=600,
        bbox_inches="tight",
        facecolor="white",
    )
    fig.savefig(
        OUTPUT_DIR / f"{filename_stem}.pdf",
        bbox_inches="tight",
        facecolor="white",
    )
    plt.close(fig)


def save_combined_figure(
    summary: dict[str, tuple[pd.DataFrame, pd.DataFrame]],
) -> None:
    """保存 SUNNY 和 HEATWAVE 上下排列的综合图。"""
    fig = plt.figure(figsize=(13.5, 14.0), dpi=180)
    grid = fig.add_gridspec(nrows=2, ncols=1, hspace=0.46)

    for row_index, target in enumerate(TARGETS):
        outer_df, inner_df = summary[target]
        draw_donut_and_bar(
            fig=fig,
            grid_cell=grid[row_index, 0],
            outer_df=outer_df,
            inner_df=inner_df,
            target=target,
        )

    fig.suptitle(
        "Grouped SHAP importance under sunny and heatwave conditions",
        fontsize=18,
        fontweight="bold",
        y=0.992,
    )

    fig.subplots_adjust(
        left=0.05,
        right=0.97,
        top=0.95,
        bottom=0.08,
    )

    filename_stem = "SUNNY_HEATWAVE_SHAP_grouped_double_donut_bar"
    fig.savefig(
        OUTPUT_DIR / f"{filename_stem}.png",
        dpi=600,
        bbox_inches="tight",
        facecolor="white",
    )
    fig.savefig(
        OUTPUT_DIR / f"{filename_stem}.pdf",
        bbox_inches="tight",
        facecolor="white",
    )
    plt.close(fig)


# =============================================================================
# 7. 结果表输出
# =============================================================================
def save_summary_tables(
    outer_all_df: pd.DataFrame,
    inner_all_df: pd.DataFrame,
) -> None:
    """保存分类汇总 CSV 和 Excel。"""
    summary_all_df = pd.concat(
        [outer_all_df, inner_all_df],
        ignore_index=True,
    )

    summary_all_df.to_csv(
        OUTPUT_DIR / "SHAP_grouped_importance_summary.csv",
        index=False,
        encoding="utf-8-sig",
    )

    with pd.ExcelWriter(
        OUTPUT_DIR / "SHAP_grouped_importance_summary.xlsx",
        engine="openpyxl",
    ) as writer:
        outer_all_df.to_excel(
            writer,
            sheet_name="outer_groups",
            index=False,
        )
        inner_all_df.to_excel(
            writer,
            sheet_name="inner_groups",
            index=False,
        )
        summary_all_df.to_excel(
            writer,
            sheet_name="all_groups",
            index=False,
        )


# =============================================================================
# 8. 主程序
# =============================================================================
def main() -> None:
    print("=" * 80)
    print("读取 RF 正式模型的 SHAP 重要性结果")
    print("=" * 80)
    print(f"输入文件：{INPUT_PATH}")
    print(f"输入工作表：{INPUT_SHEET}")

    validate_group_definitions()
    shap_df = read_shap_importance()
    validate_input(shap_df)

    print(
        f"\n读取完成：{shap_df.shape[0]} 行，"
        f"{shap_df.shape[1]} 列"
    )
    print("字段、天气情景和 90 m 变量检查通过。")

    summary: dict[str, tuple[pd.DataFrame, pd.DataFrame]] = {}
    all_outer: list[pd.DataFrame] = []
    all_inner: list[pd.DataFrame] = []

    for target in TARGETS:
        outer_df, inner_df = aggregate_groups(shap_df, target)
        summary[target] = (outer_df, inner_df)
        all_outer.append(outer_df)
        all_inner.append(inner_df)

        print("\n" + "-" * 80)
        print(TARGET_FULL_LABELS[target])
        print("-" * 80)
        print("\n外环分类：")
        print(
            outer_df[
                ["Group", "MeanAbsSHAP_sum", "Percentage"]
            ].to_string(index=False)
        )
        print("\n内环分类：")
        print(
            inner_df[
                ["Group", "MeanAbsSHAP_sum", "Percentage"]
            ].to_string(index=False)
        )

        save_single_figure(outer_df, inner_df, target)

    outer_all_df = pd.concat(all_outer, ignore_index=True)
    inner_all_df = pd.concat(all_inner, ignore_index=True)

    save_summary_tables(outer_all_df, inner_all_df)
    save_combined_figure(summary)

    print("\n" + "=" * 80)
    print("全部完成")
    print("=" * 80)
    print(f"结果目录：{OUTPUT_DIR}")


if __name__ == "__main__":
    main()


读取 RF 正式模型的 SHAP 重要性结果
输入文件：E:\excel\FILES\博士申请\第二篇论文\脚本\Python\importance\RF_formal_model_results.xlsx
输入工作表：SHAP_importance

读取完成：34 行，5 列
字段、天气情景和 90 m 变量检查通过。

--------------------------------------------------------------------------------
Sunny-day condition
--------------------------------------------------------------------------------

外环分类：
              Group  MeanAbsSHAP_sum  Percentage
External landscapes         0.851787   39.369335
Internal landscapes         0.949762   43.897703
  Control variables         0.362031   16.732962

内环分类：
                      Group  MeanAbsSHAP_sum  Percentage
External 2D characteristics         0.725428   33.529065
External 3D characteristics         0.126359    5.840270
Internal 2D characteristics         0.578584   26.741993
Internal 3D characteristics         0.371178   17.155711
          Control variables         0.362031   16.732962

--------------------------------------------------------------------------------
Heatwave condition
-

In [1]:
# -*- coding: utf-8 -*-
"""
根据 RF 正式模型结果中的 TreeSHAP 重要性，绘制双层环形图和分类占比柱状图。

输入文件：
E:/excel/FILES/博士申请/第二篇论文/脚本/Python/importance/
RF_formal_model_results.xlsx

输入工作表：SHAP_importance

原始配色：
- 外环：Matplotlib tab10 前三种颜色
- 内环及柱状图：Matplotlib Pastel1 前五种颜色

输出：
1. SUNNY 单独图（PNG、PDF）
2. HEATWAVE 单独图（PNG、PDF）
3. 两情景上下排列综合图（PNG、PDF）
4. 分类汇总 CSV 和 Excel
"""

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch, Wedge

warnings.filterwarnings("ignore")


# =============================================================================
# 1. 路径与参数
# =============================================================================
INPUT_PATH = Path(
    r"E:\excel\FILES\博士申请\第二篇论文\脚本\Python\importance"
    r"\RF_formal_model_results.xlsx"
)
INPUT_SHEET = "SHAP_importance"

OUTPUT_DIR = Path(
    r"E:\excel\FILES\博士申请\第二篇论文\脚本\Python\importance\cycle"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGETS = ["SUNNY", "HEATWAVE"]

TARGET_LABELS = {
    "SUNNY": "Sunny",
    "HEATWAVE": "Heatwave",
}

TARGET_FULL_LABELS = {
    "SUNNY": "Sunny-day condition",
    "HEATWAVE": "Heatwave condition",
}

# 重点变量：数据表中的实际字段名及图中显示名称
TARGET_VARIABLE = "TCC_b90m"
TARGET_VARIABLE_LABEL = "TCC_buffer"
TARGET_INNER_GROUP = "External 2D characteristics"


# =============================================================================
# 2. 变量分类
# =============================================================================
OUTER_GROUPS = {
    "External landscapes": [
        "TCC_b90m", "GCI_b90m", "BCI_b90m", "WCI_b90m",
        "RCI_b90m", "BAH_b90m", "BHSD_b90m",
    ],
    "Internal landscapes": [
        "TCC", "GCI", "BCI", "WCI", "RCI", "BAH", "BHSD",
    ],
    "Control variables": [
        "Shape_Area", "jungong", "DIST",
    ],
}

INNER_GROUPS = {
    "External 2D characteristics": [
        "TCC_b90m", "GCI_b90m", "BCI_b90m", "WCI_b90m", "RCI_b90m",
    ],
    "External 3D characteristics": [
        "BAH_b90m", "BHSD_b90m",
    ],
    "Internal 2D characteristics": [
        "TCC", "GCI", "BCI", "WCI", "RCI",
    ],
    "Internal 3D characteristics": [
        "BAH", "BHSD",
    ],
    "Control variables": [
        "Shape_Area", "jungong", "DIST",
    ],
}

BAR_LABELS = {
    "External 2D characteristics": "External 2D",
    "External 3D characteristics": "External 3D",
    "Internal 2D characteristics": "Internal 2D",
    "Internal 3D characteristics": "Internal 3D",
    "Control variables": "Control variables",
}


# =============================================================================
# 3. 原始配色方案
# =============================================================================
def get_plot_colors() -> tuple[list, list]:
    """返回原始代码中的 tab10 + Pastel1 配色。"""
    outer_cmap = plt.get_cmap("tab10")
    inner_cmap = plt.get_cmap("Pastel1")

    outer_colors = [outer_cmap(i) for i in range(3)]
    inner_colors = [inner_cmap(i) for i in range(5)]

    return outer_colors, inner_colors


# =============================================================================
# 4. 数据读取与检查
# =============================================================================
def read_shap_importance() -> pd.DataFrame:
    """读取 SHAP_importance 工作表并规范字段类型。"""
    if not INPUT_PATH.exists():
        raise FileNotFoundError(f"找不到输入文件：\n{INPUT_PATH}")

    try:
        data = pd.read_excel(
            INPUT_PATH,
            sheet_name=INPUT_SHEET,
            engine="openpyxl",
        )
    except ValueError as error:
        raise ValueError(
            f"Excel 文件中找不到工作表：{INPUT_SHEET}"
        ) from error

    data = data.dropna(how="all").copy()
    data.columns = data.columns.astype(str).str.strip()

    for column in ["LST", "Variable", "Variable_Label"]:
        if column in data.columns:
            data[column] = data[column].astype(str).str.strip()

    if "LST" in data.columns:
        data["LST"] = data["LST"].str.upper()

    for column in ["MeanAbsSHAP", "RelativeImportance_percent"]:
        if column in data.columns:
            data[column] = pd.to_numeric(data[column], errors="coerce")

    data.replace([np.inf, -np.inf], np.nan, inplace=True)
    return data


def validate_group_definitions() -> None:
    """检查分类字典是否存在重复或遗漏。"""
    outer_variables = [v for values in OUTER_GROUPS.values() for v in values]
    inner_variables = [v for values in INNER_GROUPS.values() for v in values]

    outer_duplicates = sorted(
        {v for v in outer_variables if outer_variables.count(v) > 1}
    )
    inner_duplicates = sorted(
        {v for v in inner_variables if inner_variables.count(v) > 1}
    )

    if outer_duplicates:
        raise ValueError(f"外环分类中存在重复变量：{outer_duplicates}")
    if inner_duplicates:
        raise ValueError(f"内环分类中存在重复变量：{inner_duplicates}")
    if set(outer_variables) != set(inner_variables):
        raise ValueError("外环与内环包含的变量集合不一致。")


def validate_input(data: pd.DataFrame) -> None:
    """检查输入字段、情景、变量及数值完整性。"""
    required_columns = {
        "LST",
        "Variable",
        "Variable_Label",
        "MeanAbsSHAP",
        "RelativeImportance_percent",
    }

    missing_columns = sorted(required_columns.difference(data.columns))
    if missing_columns:
        raise KeyError(
            "SHAP_importance 工作表缺少以下字段：\n"
            + "\n".join(missing_columns)
        )

    observed_targets = set(data["LST"].dropna().astype(str).str.upper())
    missing_targets = sorted(set(TARGETS).difference(observed_targets))
    if missing_targets:
        raise KeyError(
            "SHAP_importance 工作表缺少以下情景：\n"
            + "\n".join(missing_targets)
        )

    required_variables = {
        variable
        for variable_list in OUTER_GROUPS.values()
        for variable in variable_list
    }

    for target in TARGETS:
        target_rows = data.loc[data["LST"] == target].copy()

        duplicates = target_rows.loc[
            target_rows["Variable"].duplicated(keep=False), "Variable"
        ].unique()
        if len(duplicates) > 0:
            raise ValueError(
                f"{target} 中以下变量出现重复记录：\n"
                + "\n".join(sorted(map(str, duplicates)))
            )

        current_variables = set(target_rows["Variable"].dropna().astype(str))
        missing_variables = sorted(required_variables.difference(current_variables))
        if missing_variables:
            raise KeyError(
                f"{target} 缺少以下变量：\n"
                + "\n".join(missing_variables)
            )

    invalid_shap = data["MeanAbsSHAP"].isna() | ~np.isfinite(data["MeanAbsSHAP"])
    if invalid_shap.any():
        invalid_rows = data.loc[
            invalid_shap,
            ["LST", "Variable", "MeanAbsSHAP"],
        ]
        raise ValueError(
            "MeanAbsSHAP 中存在无法识别的值：\n"
            + invalid_rows.to_string(index=False)
        )

    if (data["MeanAbsSHAP"] < 0).any():
        raise ValueError("MeanAbsSHAP 中存在负值，请检查输入数据。")


# =============================================================================
# 5. 分类汇总
# =============================================================================
def aggregate_groups(
    data: pd.DataFrame,
    target: str,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    按 MeanAbsSHAP 求和后重新计算分类百分比，并计算 TCC_buffer 的比例。

    TCC_buffer 同时计算两个口径：
    1. 占全部变量总 SHAP 的比例；
    2. 占 External 2D characteristics 组内 SHAP 的比例。
    """
    target_data = (
        data.loc[data["LST"] == target]
        .copy()
        .set_index("Variable")
    )

    total_shap = float(target_data["MeanAbsSHAP"].sum())
    if not np.isfinite(total_shap) or total_shap <= 0:
        raise ValueError(f"{target} 的 MeanAbsSHAP 总和无效：{total_shap}")

    def make_summary(groups: dict[str, list[str]], level: str) -> pd.DataFrame:
        rows = []
        for group_name, variables in groups.items():
            group_shap = float(target_data.loc[variables, "MeanAbsSHAP"].sum())
            rows.append(
                {
                    "LST": target,
                    "Level": level,
                    "Group": group_name,
                    "MeanAbsSHAP_sum": group_shap,
                    "Percentage": group_shap / total_shap * 100,
                }
            )
        return pd.DataFrame(rows)

    outer_df = make_summary(OUTER_GROUPS, "Outer")
    inner_df = make_summary(INNER_GROUPS, "Inner")

    for level_name, result in [("外环", outer_df), ("内环", inner_df)]:
        percentage_sum = result["Percentage"].sum()
        if not np.isclose(percentage_sum, 100.0, atol=1e-6):
            raise ValueError(
                f"{target} {level_name}分类百分比之和不等于 100%："
                f"{percentage_sum:.10f}"
            )

    # -------------------------------------------------------------------------
    # TCC_buffer 单变量比例
    # -------------------------------------------------------------------------
    tcc_shap = float(target_data.loc[TARGET_VARIABLE, "MeanAbsSHAP"])
    external_2d_shap = float(
        target_data.loc[
            INNER_GROUPS[TARGET_INNER_GROUP],
            "MeanAbsSHAP",
        ].sum()
    )

    tcc_total_percent = tcc_shap / total_shap * 100
    tcc_group_percent = tcc_shap / external_2d_shap * 100

    target_variable_df = pd.DataFrame(
        [
            {
                "LST": target,
                "Variable": TARGET_VARIABLE,
                "DisplayLabel": TARGET_VARIABLE_LABEL,
                "MeanAbsSHAP": tcc_shap,
                "Percentage_of_total": tcc_total_percent,
                "ParentGroup": TARGET_INNER_GROUP,
                "Percentage_within_parent_group": tcc_group_percent,
            }
        ]
    )

    return outer_df, inner_df, target_variable_df


# =============================================================================
# 6. 绘图
# =============================================================================
def draw_donut_and_bar(
    fig: plt.Figure,
    grid_cell,
    outer_df: pd.DataFrame,
    inner_df: pd.DataFrame,
    target_variable_df: pd.DataFrame,
    target: str,
) -> None:
    """
    在一个 GridSpec 区域内绘制双层环形图和水平柱状图。

    TCC_buffer 使用黑色斜线覆盖层突出显示，不改变原始配色：
    - 环形图：在 External 2D 内环扇区中标出其占全部 SHAP 的比例；
    - 柱状图：在 External 2D 柱内叠加 TCC_buffer 对应长度。
    """
    subgrid = grid_cell.subgridspec(
        1,
        2,
        width_ratios=[1.18, 1.00],
        wspace=0.35,
    )

    ax_donut = fig.add_subplot(subgrid[0, 0])
    ax_bar = fig.add_subplot(subgrid[0, 1])

    outer_values = outer_df["Percentage"].to_numpy(dtype=float)
    outer_names = outer_df["Group"].tolist()
    inner_values = inner_df["Percentage"].to_numpy(dtype=float)
    inner_names = inner_df["Group"].tolist()

    tcc_total_percent = float(
        target_variable_df["Percentage_of_total"].iloc[0]
    )
    tcc_group_percent = float(
        target_variable_df["Percentage_within_parent_group"].iloc[0]
    )

    # 使用原始配色
    outer_colors, inner_colors = get_plot_colors()

    outer_labels = [
        f"{name}\n{value:.1f}%"
        for name, value in zip(outer_names, outer_values)
    ]

    ax_donut.pie(
        outer_values,
        radius=1.00,
        labels=outer_labels,
        labeldistance=1.13,
        colors=outer_colors,
        startangle=90,
        counterclock=False,
        wedgeprops={
            "width": 0.27,
            "edgecolor": "white",
            "linewidth": 1.5,
        },
        textprops={
            "fontsize": 10.5,
            "fontweight": "bold",
            "ha": "center",
        },
    )

    ax_donut.pie(
        inner_values,
        radius=0.71,
        labels=None,
        colors=inner_colors,
        startangle=90,
        counterclock=False,
        wedgeprops={
            "width": 0.27,
            "edgecolor": "white",
            "linewidth": 1.5,
        },
    )

    # -------------------------------------------------------------------------
    # 环形图：TCC_buffer 高亮覆盖层
    # External 2D 是内环第一类，因此其扇区从 90° 开始顺时针展开。
    # TCC_buffer 的圆弧长度直接对应其“占全部 SHAP 的比例”。
    # -------------------------------------------------------------------------
    tcc_angle = tcc_total_percent / 100.0 * 360.0
    tcc_theta1 = 90.0 - tcc_angle
    tcc_theta2 = 90.0

    tcc_wedge = Wedge(
        center=(0, 0),
        r=0.71,
        theta1=tcc_theta1,
        theta2=tcc_theta2,
        width=0.27,
        facecolor="none",
        edgecolor="black",
        linewidth=1.15,
        hatch="////",
        zorder=5,
    )
    ax_donut.add_patch(tcc_wedge)

    # 标注箭头指向 TCC_buffer 扇区中点
    mid_angle = np.deg2rad((tcc_theta1 + tcc_theta2) / 2.0)
    arrow_radius = 0.58
    arrow_xy = (
        arrow_radius * np.cos(mid_angle),
        arrow_radius * np.sin(mid_angle),
    )

    # 根据扇区所在侧自动选择文字位置，减少遮挡
    text_x = 1.08 if arrow_xy[0] >= 0 else -1.08
    text_y = max(min(arrow_xy[1] + 0.18, 0.82), -0.72)
    text_ha = "left" if text_x > 0 else "right"

    ax_donut.annotate(
        f"{TARGET_VARIABLE_LABEL}: {tcc_total_percent:.1f}%\n"
        f"({tcc_group_percent:.1f}% of External 2D)",
        xy=arrow_xy,
        xytext=(text_x, text_y),
        ha=text_ha,
        va="center",
        fontsize=9.5,
        fontweight="bold",
        arrowprops={
            "arrowstyle": "-",
            "color": "black",
            "linewidth": 1.1,
            "connectionstyle": "arc3,rad=0.12",
        },
        bbox={
            "boxstyle": "round,pad=0.28",
            "facecolor": "white",
            "edgecolor": "black",
            "linewidth": 0.8,
            "alpha": 0.95,
        },
        zorder=8,
    )

    ax_donut.text(
        0,
        0.05,
        TARGET_LABELS[target],
        ha="center",
        va="center",
        fontsize=15,
        fontweight="bold",
    )
    ax_donut.text(
        0,
        -0.12,
        "SHAP",
        ha="center",
        va="center",
        fontsize=10,
        fontweight="bold",
        color="#666666",
    )
    ax_donut.set_aspect("equal")

    legend_handles = [
        Patch(
            facecolor=inner_colors[i],
            edgecolor="white",
            label=f"{name}: {inner_values[i]:.1f}%",
        )
        for i, name in enumerate(inner_names)
    ]

    # 在图例中补充 TCC_buffer 的斜线符号
    legend_handles.append(
        Patch(
            facecolor="white",
            edgecolor="black",
            hatch="////",
            label=(
                f"{TARGET_VARIABLE_LABEL}: {tcc_total_percent:.1f}% total; "
                f"{tcc_group_percent:.1f}% of External 2D"
            ),
        )
    )

    ax_donut.legend(
        handles=legend_handles,
        title="Inner-ring categories",
        loc="upper center",
        bbox_to_anchor=(0.50, -0.08),
        ncol=1,
        frameon=False,
        fontsize=9.0,
        title_fontsize=10.2,
    )

    # 与原代码一致：反转顺序，使第一类显示在最上方
    bar_df = inner_df.copy()
    bar_df["DisplayLabel"] = bar_df["Group"].map(BAR_LABELS)
    bar_df = bar_df.iloc[::-1].reset_index(drop=True)
    bar_colors = inner_colors[::-1]

    bars = ax_bar.barh(
        bar_df["DisplayLabel"],
        bar_df["Percentage"],
        color=bar_colors,
        edgecolor="black",
        linewidth=0.65,
        alpha=0.92,
        height=0.66,
    )

    max_percent = float(bar_df["Percentage"].max())
    x_limit = max(max_percent * 1.50, max_percent + 8.0)
    ax_bar.set_xlim(0, x_limit)

    for bar, value in zip(bars, bar_df["Percentage"]):
        ax_bar.text(
            bar.get_width() + max_percent * 0.025,
            bar.get_y() + bar.get_height() / 2,
            f"{value:.1f}%",
            va="center",
            ha="left",
            fontsize=10.5,
            fontweight="bold",
        )

    # -------------------------------------------------------------------------
    # 柱状图：在 External 2D 柱体内部叠加 TCC_buffer 所占长度
    # 这里的宽度仍使用其“占全部 SHAP 的比例”，因此可与整根柱直接比较。
    # -------------------------------------------------------------------------
    external_bar_index = bar_df.index[
        bar_df["Group"] == TARGET_INNER_GROUP
    ][0]
    external_bar = bars[external_bar_index]

    ax_bar.barh(
        y=external_bar.get_y() + external_bar.get_height() / 2,
        width=tcc_total_percent,
        height=external_bar.get_height(),
        left=0,
        facecolor="none",
        edgecolor="black",
        linewidth=1.15,
        hatch="////",
        zorder=5,
    )

    # 优先将标签放在柱体内部；比例过小时移到柱体上方
    if tcc_total_percent >= 7.0:
        label_x = tcc_total_percent / 2.0
        label_y = external_bar.get_y() + external_bar.get_height() / 2
        label_ha = "center"
        label_va = "center"
    else:
        label_x = tcc_total_percent + max_percent * 0.02
        label_y = external_bar.get_y() + external_bar.get_height() * 0.82
        label_ha = "left"
        label_va = "bottom"

    ax_bar.text(
        label_x,
        label_y,
        f"{TARGET_VARIABLE_LABEL}\n{tcc_total_percent:.1f}% total\n"
        f"{tcc_group_percent:.1f}% of group",
        ha=label_ha,
        va=label_va,
        fontsize=8.7,
        fontweight="bold",
        zorder=7,
        bbox={
            "boxstyle": "round,pad=0.20",
            "facecolor": "white",
            "edgecolor": "none",
            "alpha": 0.78,
        },
    )

    ax_bar.set_xlabel(
        "Relative importance (%)",
        fontsize=12,
        fontweight="bold",
        labelpad=8,
    )
    ax_bar.set_ylabel("")
    ax_bar.set_title(
        "Grouped SHAP importance",
        fontsize=13,
        fontweight="bold",
        pad=14,
    )

    ax_bar.tick_params(
        axis="both",
        labelsize=10.5,
        width=1.1,
        length=4.5,
        direction="out",
    )

    for tick_label in ax_bar.get_xticklabels() + ax_bar.get_yticklabels():
        tick_label.set_fontweight("bold")

    ax_bar.spines["top"].set_visible(False)
    ax_bar.spines["right"].set_visible(False)
    ax_bar.spines["left"].set_linewidth(1.15)
    ax_bar.spines["bottom"].set_linewidth(1.15)
    ax_bar.grid(False)


def save_single_figure(
    outer_df: pd.DataFrame,
    inner_df: pd.DataFrame,
    target_variable_df: pd.DataFrame,
    target: str,
) -> None:
    """保存一个天气情景的单独图。"""
    fig = plt.figure(figsize=(13.2, 7.0), dpi=180)
    grid = fig.add_gridspec(1, 1)

    draw_donut_and_bar(
        fig=fig,
        grid_cell=grid[0, 0],
        outer_df=outer_df,
        inner_df=inner_df,
        target_variable_df=target_variable_df,
        target=target,
    )

    fig.suptitle(
        f"{TARGET_FULL_LABELS[target]}: grouped SHAP importance",
        fontsize=16,
        fontweight="bold",
        y=0.985,
    )

    fig.subplots_adjust(
        left=0.05,
        right=0.97,
        top=0.89,
        bottom=0.22,
    )

    filename_stem = f"{target}_SHAP_grouped_double_donut_bar"
    fig.savefig(
        OUTPUT_DIR / f"{filename_stem}.png",
        dpi=600,
        bbox_inches="tight",
        facecolor="white",
    )
    fig.savefig(
        OUTPUT_DIR / f"{filename_stem}.pdf",
        bbox_inches="tight",
        facecolor="white",
    )
    plt.close(fig)


def save_combined_figure(
    summary: dict[
        str,
        tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame],
    ],
) -> None:
    """保存 SUNNY 和 HEATWAVE 上下排列的综合图。"""
    fig = plt.figure(figsize=(13.5, 14.0), dpi=180)
    grid = fig.add_gridspec(nrows=2, ncols=1, hspace=0.46)

    for row_index, target in enumerate(TARGETS):
        outer_df, inner_df, target_variable_df = summary[target]
        draw_donut_and_bar(
            fig=fig,
            grid_cell=grid[row_index, 0],
            outer_df=outer_df,
            inner_df=inner_df,
            target_variable_df=target_variable_df,
            target=target,
        )

    fig.suptitle(
        "Grouped SHAP importance under sunny and heatwave conditions",
        fontsize=18,
        fontweight="bold",
        y=0.992,
    )

    fig.subplots_adjust(
        left=0.05,
        right=0.97,
        top=0.95,
        bottom=0.08,
    )

    filename_stem = "SUNNY_HEATWAVE_SHAP_grouped_double_donut_bar"
    fig.savefig(
        OUTPUT_DIR / f"{filename_stem}.png",
        dpi=600,
        bbox_inches="tight",
        facecolor="white",
    )
    fig.savefig(
        OUTPUT_DIR / f"{filename_stem}.pdf",
        bbox_inches="tight",
        facecolor="white",
    )
    plt.close(fig)


# =============================================================================
# 7. 结果表输出
# =============================================================================
def save_summary_tables(
    outer_all_df: pd.DataFrame,
    inner_all_df: pd.DataFrame,
    target_variable_all_df: pd.DataFrame,
) -> None:
    """保存分类汇总 CSV 和 Excel。"""
    summary_all_df = pd.concat(
        [outer_all_df, inner_all_df],
        ignore_index=True,
    )

    summary_all_df.to_csv(
        OUTPUT_DIR / "SHAP_grouped_importance_summary.csv",
        index=False,
        encoding="utf-8-sig",
    )

    target_variable_all_df.to_csv(
        OUTPUT_DIR / "TCC_buffer_importance_summary.csv",
        index=False,
        encoding="utf-8-sig",
    )

    with pd.ExcelWriter(
        OUTPUT_DIR / "SHAP_grouped_importance_summary.xlsx",
        engine="openpyxl",
    ) as writer:
        outer_all_df.to_excel(
            writer,
            sheet_name="outer_groups",
            index=False,
        )
        inner_all_df.to_excel(
            writer,
            sheet_name="inner_groups",
            index=False,
        )
        summary_all_df.to_excel(
            writer,
            sheet_name="all_groups",
            index=False,
        )
        target_variable_all_df.to_excel(
            writer,
            sheet_name="TCC_buffer",
            index=False,
        )


# =============================================================================
# 8. 主程序
# =============================================================================
def main() -> None:
    print("=" * 80)
    print("读取 RF 正式模型的 SHAP 重要性结果")
    print("=" * 80)
    print(f"输入文件：{INPUT_PATH}")
    print(f"输入工作表：{INPUT_SHEET}")

    validate_group_definitions()
    shap_df = read_shap_importance()
    validate_input(shap_df)

    print(
        f"\n读取完成：{shap_df.shape[0]} 行，"
        f"{shap_df.shape[1]} 列"
    )
    print("字段、天气情景和 90 m 变量检查通过。")

    summary: dict[
        str,
        tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame],
    ] = {}
    all_outer: list[pd.DataFrame] = []
    all_inner: list[pd.DataFrame] = []
    all_target_variable: list[pd.DataFrame] = []

    for target in TARGETS:
        outer_df, inner_df, target_variable_df = aggregate_groups(
            shap_df,
            target,
        )
        summary[target] = (outer_df, inner_df, target_variable_df)
        all_outer.append(outer_df)
        all_inner.append(inner_df)
        all_target_variable.append(target_variable_df)

        print("\n" + "-" * 80)
        print(TARGET_FULL_LABELS[target])
        print("-" * 80)
        print("\n外环分类：")
        print(
            outer_df[
                ["Group", "MeanAbsSHAP_sum", "Percentage"]
            ].to_string(index=False)
        )
        print("\n内环分类：")
        print(
            inner_df[
                ["Group", "MeanAbsSHAP_sum", "Percentage"]
            ].to_string(index=False)
        )

        print("\nTCC_buffer 重点变量：")
        print(
            target_variable_df[
                [
                    "DisplayLabel",
                    "MeanAbsSHAP",
                    "Percentage_of_total",
                    "Percentage_within_parent_group",
                ]
            ].to_string(index=False)
        )

        save_single_figure(
            outer_df,
            inner_df,
            target_variable_df,
            target,
        )

    outer_all_df = pd.concat(all_outer, ignore_index=True)
    inner_all_df = pd.concat(all_inner, ignore_index=True)
    target_variable_all_df = pd.concat(
        all_target_variable,
        ignore_index=True,
    )

    save_summary_tables(
        outer_all_df,
        inner_all_df,
        target_variable_all_df,
    )
    save_combined_figure(summary)

    print("\n" + "=" * 80)
    print("全部完成")
    print("=" * 80)
    print(f"结果目录：{OUTPUT_DIR}")


if __name__ == "__main__":
    main()

读取 RF 正式模型的 SHAP 重要性结果
输入文件：E:\excel\FILES\博士申请\第二篇论文\脚本\Python\importance\RF_formal_model_results.xlsx
输入工作表：SHAP_importance

读取完成：34 行，5 列
字段、天气情景和 90 m 变量检查通过。

--------------------------------------------------------------------------------
Sunny-day condition
--------------------------------------------------------------------------------

外环分类：
              Group  MeanAbsSHAP_sum  Percentage
External landscapes         0.851787   39.369335
Internal landscapes         0.949762   43.897703
  Control variables         0.362031   16.732962

内环分类：
                      Group  MeanAbsSHAP_sum  Percentage
External 2D characteristics         0.725428   33.529065
External 3D characteristics         0.126359    5.840270
Internal 2D characteristics         0.578584   26.741993
Internal 3D characteristics         0.371178   17.155711
          Control variables         0.362031   16.732962

TCC_buffer 重点变量：
DisplayLabel  MeanAbsSHAP  Percentage_of_total  Percentage_within_parent_group
  TCC